# Notebook 2 — Generate Embeddings

Embeds SNOMED CT preferred terms using `text-embedding-3-large` (3072-dim),
normalises to the unit sphere, and aligns with the ontological distance matrix.

**Input files** (from notebook 1)
- `{DATA_DIR}/concepts.csv`
- `{DATA_DIR}/ontological_distances.csv`

**Output files**
- `{DATA_DIR}/embeddings_raw.csv`
- `{DATA_DIR}/embeddings_normalised.csv`
- `{DATA_DIR}/concepts_filtered.csv`
- `{DATA_DIR}/ontological_distances_filtered.csv`

In [ ]:
# parameters
DATA_DIR = "../../data/embeddings-concept-openai"

## 2a. Embed preferred terms

`get_embeddings()` reused directly from `Reference_Papers/Representation-Manifolds/2-get_text_embeddings.ipynb`.

In [ ]:
import openai
import os
import pandas as pd
import numpy as np

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# --- reused from 2-get_text_embeddings.ipynb ---
def chunker(seq, size):
    return (seq[pos:pos + size] for pos in range(0, len(seq), size))

def get_embeddings(queries, model, **kwargs):
    queries = [q.replace('\n', ' ') for q in queries]
    embeddings_data = []
    for chunk in chunker(queries, 2048):
        response = client.embeddings.create(input=chunk, model=model, **kwargs)
        embeddings_data.extend(response.data)
    return pd.DataFrame([x.embedding for x in embeddings_data], index=queries)
# --- end reuse ---

In [ ]:
df_concepts = pd.read_csv(f"{DATA_DIR}/concepts.csv")
print(f"{len(df_concepts)} concepts loaded")

X_raw = get_embeddings(df_concepts["preferred_term"].tolist(), model="text-embedding-3-large")
print(f"Embedding matrix shape: {X_raw.shape}")

X_raw.to_csv(f"{DATA_DIR}/embeddings_raw.csv", index=False, header=False)
print(f"Saved {DATA_DIR}/embeddings_raw.csv")

## 2b. Normalise and align with distance matrix

In [ ]:
D_snomed = pd.read_csv(f"{DATA_DIR}/ontological_distances.csv", header=None).values

norms = np.linalg.norm(X_raw.values, axis=1)
mask  = norms > 1e-3                                       # remove zero-norm rows
X     = X_raw.values[mask] / norms[mask, np.newaxis]      # unit sphere [n, 3072]

df_concepts = df_concepts[mask].reset_index(drop=True)
D_snomed    = D_snomed[np.ix_(mask, mask)]

print(f"After filtering: {X.shape[0]} concepts, {X.shape[1]} dims")
print(f"Norm check (should be ~1.0): min={np.linalg.norm(X, axis=1).min():.6f}, max={np.linalg.norm(X, axis=1).max():.6f}")

In [ ]:
pd.DataFrame(X).to_csv(f"{DATA_DIR}/embeddings_normalised.csv", index=False, header=False)
df_concepts.to_csv(f"{DATA_DIR}/concepts_filtered.csv", index=False)
pd.DataFrame(D_snomed).to_csv(f"{DATA_DIR}/ontological_distances_filtered.csv", index=False, header=False)

print("Saved:")
print(f"  {DATA_DIR}/embeddings_normalised.csv")
print(f"  {DATA_DIR}/concepts_filtered.csv")
print(f"  {DATA_DIR}/ontological_distances_filtered.csv")